# Regressão
## Utilizar LLM, categoria e preço para chegar na caixa com menor valor possível
UTILIZAR LLM, LEVAR EM CONSIDERACAO CATEGORIA E PRECO PARA CHEGAR NA CAIXA COM O MENOR VALOR POSSIVEL

In [3]:
import pandas as pd
import numpy as np
import joblib

from sentence_transformers import SentenceTransformer
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [7]:
train = pd.read_csv(
    "../data/cubagem_40k_amazon_train.csv",
)

val = pd.read_csv(
    "../data/cubagem_40k_amazon_val.csv",
)

In [8]:
print("Treino:", len(train))
print("Validação:", len(val))

Treino: 32000
Validação: 8000


In [9]:
colunas_alvo = [
    "length_cm",
    "width_cm",
    "height_cm",
    "weight_g",
]

train = train.dropna(
    subset=colunas_alvo
)

val = val.dropna(
    subset=colunas_alvo
)

In [10]:
def coluna_segura(df, nome):
    if nome in df.columns:
        return df[nome].fillna("").astype(str)
    return ""

def criar_texto(df):

    return (
        coluna_segura(df, "title")
        + " "
        + coluna_segura(df, "description")
        + " "
        + coluna_segura(df, "main_category")
        + " "
        + coluna_segura(df, "source_category")
        + " "
    )

In [11]:
train["texto"] = criar_texto(train)
val["texto"] = criar_texto(val)

In [12]:
from huggingface_hub import login

login("have to remove the login from commit")

In [13]:
modelo_texto = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

ValueError: Unrecognized processing class in sentence-transformers/all-MiniLM-L6-v2. Can't instantiate a processor, a tokenizer, an image processor, a video processor or a feature extractor for this model. Make sure the repository contains the files of at least one of those processing classes.

In [ ]:
print("Gerando embeddings treino...")

X_train = modelo_texto.encode(
    train["texto"].tolist(),
    batch_size=64,
    show_progress_bar =True,
    normalize_embeddings=True
)
print("Gerando embeddings validação...")

X_val = modelo_texto.encode(
    val["texto"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

In [ ]:
print(X_train.shape)
print(X_val.shape)

In [ ]:
y_train_length = train["length_cm"]
y_train_width = train["width_cm"]
y_train_height = train["height_cm"]
y_train_weight = train["weight_g"]


y_val_length = val["length_cm"]
y_val_width = val["width_cm"]
y_val_height = val["height_cm"]
y_val_weight = val["weight_g"]


In [ ]:
def criar_modelo():
   
    return XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
    
modelo_length = criar_modelo()
modelo_length.fit(
    X_train,
    y_train_length
)

modelo_width = criar_modelo()
modelo_width.fit(
    X_train,
    y_train_width
)

modelo_height = criar_modelo()
modelo_height.fit(
    X_train,
    y_train_height
)

modelo_weight = criar_modelo()
modelo_weight.fit(
    X_train,
    y_train_weight
)


In [ ]:
def avaliar(nome, modelo, X, y):

    pred = modelo.predict(X)

    mae = mean_absolute_error(
        y,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y,
            pred
        )
    )

    r2 = r2_score(
        y,
        pred
    )

    print("\n" + "=" * 40)
    print(nome)
    print("=" * 40)

    print("MAE :", round(mae, 3))
    print("RMSE:", round(rmse, 3))
    print("R²  :", round(r2, 4))

In [ ]:
print("\nRESULTADOS")

avaliar(
    "Comprimento",
    modelo_length,
    X_val,
    y_val_length
)

avaliar(
    "Largura",
    modelo_width,
    X_val,
    y_val_width
)

avaliar(
    "Altura",
    modelo_height,
    X_val,
    y_val_height
)


avaliar(
    "peso",
    modelo_weight,
    X_val,
    y_val_weight
)



In [ ]:
joblib.dump(
    modelo_length,
    "modelo_length.pkl"
)

joblib.dump(
    modelo_width,
    "modelo_width.pkl"
)

joblib.dump(
    modelo_height,
    "modelo_height.pkl"
)

joblib.dump(
    modelo_weight,
    "modelo_weight.pkl"
)


print("\nModelos salvos.")

In [ ]:
def calcular_densidade(
    comprimento,
    largura,
    altura,
    peso_g
):

    volume = (
        comprimento
        * largura
        * altura
    )

    if volume <= 0:
        return 0

    return peso_g / volume

In [ ]:
def validar_densidade(
    densidade
):

    return (
        0.005
        <= densidade
        <=
        5
    )

In [ ]:
def peso_cubado(
    comprimento,
    largura,
    altura,
    fator=6000
):

    return (
        comprimento
        * largura
        * altura
    ) / fator

In [ ]:
EMBALAGENS = {

    "Caixa P":
        (20, 15, 10),

    "Caixa M":
        (35, 25, 20),

    "Caixa G":
        (50, 40, 30),

    "Caixa GG":
        (70, 60, 50)

}

In [ ]:
def sugerir_embalagem(
    comprimento,
    largura,
    altura
):

    for nome, dims in EMBALAGENS.items():

        c, l, a = dims

        if (
            comprimento <= c
            and largura <= l
            and altura <= a
        ):
            return nome

    return "Embalagem Especial"

In [ ]:
def prever_produto(
    titulo,
    descricao="",
    main_category="",
    source_category="",
    categories=""
):

    partes = [
        titulo,
        descricao,
        main_category,
        source_category,
        categories
    ]

    texto = " ".join(
        str(x).strip()
        for x in partes
        if str(x).strip()
    )

    embedding = modelo_texto.encode(
        [texto],
        normalize_embeddings=True
    )


    comprimento = modelo_length.predict(embedding)[0]
    largura = modelo_width.predict(embedding)[0]
    altura = modelo_height.predict(embedding)[0]
    peso = modelo_weight.predict(embedding)[0]

    comprimento = max(1, comprimento)
    largura = max(1, largura)
    altura = max(1, altura)
    peso = max(1, peso)

    dimensoes = sorted(
        [comprimento, largura, altura],
        reverse=True
    )

    comprimento, largura, altura = dimensoes

    return {
        "comprimento": round(comprimento, 2),
        "largura": round(largura, 2),
        "altura": round(altura, 2),
        "peso": round(peso, 2)
    }

In [ ]:
while True:

    produto = input("\nProduto (ou sair): ")#.lower()

    if produto.lower() == "sair":
        break

    resultado = prever_produto(produto)

    print(resultado)